<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes/blob/main/Script_Principal_De_ejecucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from typing import List, Tuple

# ==============================
# Módulo 1: Validación Dimensional
# ==============================
def validar_dimensiones(cargas: List[List[float]], capacidades: List[List[float]]) -> bool:
    """
    Verifica que las matrices de cargas y capacidades sean coherentes.
    """
    if len(cargas) < 2 or len(capacidades) < 2:
        return False
    if len(cargas) != len(capacidades):
        return False
    for fila_c, fila_cap in zip(cargas, capacidades):
        if len(fila_c) != len(fila_cap):
            return False
        for peso, cap in zip(fila_c, fila_cap):
            if peso < 0 or cap <= 0:
                return False
    return True


# ==============================
# Módulo 2: Ocupación y Sobrecarga
# ==============================
def calcular_ocupacion(cargas: List[List[float]], capacidades: List[List[float]]) -> Tuple[List[List[float]], List[Tuple[int,int]]]:
    """
    Calcula matriz de porcentajes de ocupación y lista de celdas sobrecargadas.
    """
    N, M = len(cargas), len(cargas[0])
    matriz_ocupacion = [[0]*M for _ in range(N)]
    sobrecargadas = []
    for i in range(N):
        for j in range(M):
            porcentaje = (cargas[i][j] / capacidades[i][j]) * 100
            matriz_ocupacion[i][j] = porcentaje
            if porcentaje > 100:
                sobrecargadas.append((i, j))
    return matriz_ocupacion, sobrecargadas


# ==============================
# Módulo 3: Balance y Simetría
# ==============================
def evaluar_balance(cargas: List[List[float]], tolerancia: float) -> Tuple[List[float], float, bool]:
    """
    Evalúa el balance lateral y longitudinal.
    """
    N, M = len(cargas), len(cargas[0])
    pesos_fila = [sum(fila) for fila in cargas]

    mitad = M // 2
    if M % 2 == 0:
        izquierda = sum(cargas[i][j] for i in range(N) for j in range(mitad))
        derecha = sum(cargas[i][j] for i in range(N) for j in range(mitad, M))
    else:
        izquierda = sum(cargas[i][j] for i in range(N) for j in range(mitad))
        derecha = sum(cargas[i][j] for i in range(N) for j in range(mitad+1, M))

    desbalance = abs(izquierda - derecha)
    balance_ok = desbalance <= tolerancia

    return pesos_fila, desbalance, balance_ok


# ==============================
# Módulo 4: Submatriz Crítica
# ==============================
def submatriz_critica(matriz_ocupacion: List[List[float]], k: int, p: int) -> List[List[float]]:
    """
    Extrae la submatriz k x p con mayor promedio de ocupación.
    """
    N, M = len(matriz_ocupacion), len(matriz_ocupacion[0])
    mejor_promedio = -1
    mejor_submatriz = None

    for i in range(N - k + 1):
        for j in range(M - p + 1):
            sub = [fila[j:j+p] for fila in matriz_ocupacion[i:i+k]]
            promedio = sum(sum(fila) for fila in sub) / (k*p)
            if promedio > mejor_promedio:
                mejor_promedio = promedio
                mejor_submatriz = sub
    return mejor_submatriz


# ==============================
# Script Principal de Ejecución
# ==============================
if __name__ == "__main__":
    # Ejemplo con matriz 5x4
    cargas = [
        [500, 600, 700, 800],
        [400, 300, 200, 100],
        [250, 350, 450, 550],
        [600, 500, 400, 300],
        [150, 250, 350, 450]
    ]
    capacidades = [
        [600, 600, 800, 900],
        [500, 400, 300, 200],
        [300, 400, 500, 600],
        [700, 600, 500, 400],
        [200, 300, 400, 500]
    ]

    print("=== VALIDACIÓN DE MATRICES ===")
    print("Resultado:", validar_dimensiones(cargas, capacidades))

    print("\n=== CÁLCULO DE OCUPACIÓN ===")
    ocupacion, sobrecargadas = calcular_ocupacion(cargas, capacidades)
    for fila in ocupacion:
        print(["{:.2f}%".format(val) for val in fila])
    print("Celdas sobrecargadas:", sobrecargadas)

    print("\n=== EVALUACIÓN DE BALANCE ===")
    pesos_fila, desbalance, balance_ok = evaluar_balance(cargas, tolerancia=300)
    for i, peso in enumerate(pesos_fila, start=1):
        print(f"Fila {i}: {peso:.2f} kg")
    print(f"Desbalance lateral: {desbalance:.2f} kg")
    print("Balance aprobado:", "Sí" if balance_ok else "No")

    print("\n=== SUBMATRIZ CRÍTICA ===")
    sub = submatriz_critica(ocupacion, 2, 2)
    for fila in sub:
        print(["{:.2f}%".format(val) for val in fila])


=== VALIDACIÓN DE MATRICES ===
Resultado: True

=== CÁLCULO DE OCUPACIÓN ===
['83.33%', '100.00%', '87.50%', '88.89%']
['80.00%', '75.00%', '66.67%', '50.00%']
['83.33%', '87.50%', '90.00%', '91.67%']
['85.71%', '83.33%', '80.00%', '75.00%']
['75.00%', '83.33%', '87.50%', '90.00%']
Celdas sobrecargadas: []

=== EVALUACIÓN DE BALANCE ===
Fila 1: 2600.00 kg
Fila 2: 1000.00 kg
Fila 3: 1600.00 kg
Fila 4: 1800.00 kg
Fila 5: 1200.00 kg
Desbalance lateral: 400.00 kg
Balance aprobado: No

=== SUBMATRIZ CRÍTICA ===
['87.50%', '90.00%']
['83.33%', '80.00%']
